In [1]:
"""
B2. Uneg Gate Verification and Tabular Reporting
---------------------------------------------

This script validates the Uneg(k) quantum gate by preparing computational
basis states |a⟩ (encoded in little-endian order), applying the Uneg gate,
and extracting the most-likely output basis state from the resulting
statevector.

For each test case, it computes:
    - input integer a and its k-bit representation,
    - normalized input value x_in = a / 2^k,
    - measured output bitstring and integer value,
    - normalized output value x_out,
    - expected value 2^k - a and corresponding bitstring,
    - maximum measurement probability p_max,
    - PASS/FAIL status.

Results are displayed in a formatted box-style ASCII table.
Batch tests additionally print a summary including total passes,
failures, and the minimum maximum-probability observed.
"""

import random
from typing import Any, Dict, List, Optional

from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

import _init_path
from quantum.unitary_vr import uneg


# =============================================================================
# Box-table formatting utilities
# =============================================================================
def _fmt_float(x: Any, ndigits: int = 10) -> str:
    """Format a value as a floating-point string for tabular display.

    This routine converts a value to a formatted floating-point string
    with a fixed number of decimal digits.

    Args:
        x: Value to format.
        ndigits: Number of digits after the decimal point.

    Returns:
        A formatted floating-point string. Returns an empty string if
        the value is None.

    Raises:
        ValueError: If ndigits is negative.
    """
    if ndigits < 0:
        raise ValueError("ndigits must be >= 0.")
    if x is None:
        return ""
    try:
        return f"{float(x): .{ndigits}f}"
    except Exception:
        return str(x)


def _fmt_int(x: Any) -> str:
    """Format a value as an integer string for tabular display.

    This routine converts a value to an integer string representation.

    Args:
        x: Value to format.

    Returns:
        A string containing the integer representation of the value.
        Returns an empty string if the value is None.

    Raises:
        None.
    """
    if x is None:
        return ""
    try:
        return str(int(x))
    except Exception:
        return str(x)


def _fmt_bool_status(x: Any) -> str:
    """Format a value as PASS or FAIL for tabular display.

    Args:
        x: Boolean-like value indicating test success.

    Returns:
        "PASS" if the value evaluates to True, otherwise "FAIL".
        Returns an empty string if the value is None.

    Raises:
        None.
    """
    if x is None:
        return ""
    return "PASS" if bool(x) else "FAIL"


def print_uneg_results_table(rows: List[dict], title: Optional[str] = None) -> None:
    """Print a box-style ASCII table summarizing Uneg test results.

    This routine renders a formatted table containing input parameters,
    output values, expected values, probabilities, and pass/fail status
    for one or more Uneg test cases.

    Args:
        rows: List of dictionaries containing test result fields.
        title: Optional title printed above the table.

    Returns:
        None.

    Raises:
        ValueError: If rows is not a list of dictionaries.
    """
    if not isinstance(rows, list) or any(not isinstance(r, dict) for r in rows):
        raise ValueError("rows must be a list of dictionaries.")

    headers = [
        "#", "k", "a_in", "a_bits", "x_in", "out_bits",
        "v_out", "x_out", "exp_v", "exp_bits", "p_max", "status",
    ]
    aligns = ["R"] * len(headers)

    table_rows: List[List[str]] = []
    for i, r in enumerate(rows, 1):
        table_rows.append(
            [
                str(i),
                _fmt_int(r.get("k")),
                _fmt_int(r.get("a_in")),
                str(r.get("a_in_bits", "")),
                _fmt_float(r.get("x_in"), 10),
                str(r.get("bitstr_out", "")),
                _fmt_int(r.get("v_out")),
                _fmt_float(r.get("x_out"), 10),
                _fmt_int(r.get("expected_v")),
                str(r.get("expected_bits", "")),
                _fmt_float(r.get("p_max"), 10),
                _fmt_bool_status(r.get("pass")),
            ]
        )

    widths = [len(h) for h in headers]
    for row in table_rows:
        for j, cell in enumerate(row):
            widths[j] = max(widths[j], len(cell))

    TL, TR, BL, BR = "┌", "┐", "└", "┘"
    H, V = "─", "│"
    TJ, BJ, LJ, RJ, CJ = "┬", "┴", "├", "┤", "┼"

    def hline(left: str, mid: str, right: str) -> str:
        return left + mid.join(H * (w + 2) for w in widths) + right

    def format_cell(text: str, w: int, align: str, header: bool = False) -> str:
        if header:
            return f" {text:^{w}} "
        if align == "R":
            return f" {text:>{w}} "
        return f" {text:<{w}} "

    def render_row(row: List[str], header: bool = False) -> str:
        return V + V.join(
            format_cell(row[j], widths[j], aligns[j], header=header)
            for j in range(len(headers))
        ) + V

    if title:
        print(title)

    print(hline(TL, TJ, TR))
    print(render_row(headers, header=True))
    print(hline(LJ, CJ, RJ))
    for row in table_rows:
        print(render_row(row))
    print(hline(BL, BJ, BR))


# =============================================================================
# Core test logic
# =============================================================================
def int_to_bitstring(a: int, k: int) -> str:
    """Convert an integer to a k-bit binary string in MSB-to-LSB order.

    Args:
        a: Integer value to convert.
        k: Number of bits in the output string.

    Returns:
        A zero-padded binary string of length k.

    Raises:
        None.
    """
    return format(a, f"0{k}b")


def run_uneg_once(a: int, k: int, gate=None) -> Dict[str, Any]:
    """Execute a single Uneg(k) test instance.

    This routine prepares the computational basis state |a⟩ in
    little-endian order, applies the Uneg gate, extracts the most
    probable output basis state, and computes expected values
    and validation metrics.

    Args:
        a: Integer input value with 0 < a < 2^k.
        k: Number of qubits.
        gate: Optional preconstructed Uneg gate instance.

    Returns:
        A dictionary containing input data, output data, expected values,
        probabilities, and pass/fail status.

    Raises:
        ValueError: If a is not in the range (0, 2^k).
    """
    if not (0 < a < 2**k):
        raise ValueError("Require 0 < a < 2^k so x in (0,1) is representable.")

    if gate is None:
        gate = uneg(k).to_gate()

    qc = QuantumCircuit(k)

    for i in range(k):
        if (a >> i) & 1:
            qc.x(i)

    qc.append(gate, list(range(k)))

    sv = Statevector.from_instruction(qc)
    probs = sv.probabilities_dict()

    bitstr = max(probs, key=probs.get)
    p_max = probs[bitstr]

    v_out = int(bitstr, 2)
    expected = 2**k - a

    return {
        "k": k,
        "a_in": a,
        "a_in_bits": int_to_bitstring(a, k),
        "x_in": a / (2**k),
        "bitstr_out": bitstr,
        "v_out": v_out,
        "x_out": v_out / (2**k),
        "expected_v": expected,
        "expected_bits": int_to_bitstring(expected, k),
        "p_max": p_max,
        "pass": (v_out == expected),
    }


def run_single_test(a: int, k: int = 6) -> Dict[str, Any]:
    """Run a single Uneg test and print the formatted results table.

    Args:
        a: Integer input value.
        k: Number of qubits.

    Returns:
        A dictionary containing the test result data.

    Raises:
        None.
    """
    row = run_uneg_once(a, k)
    print_uneg_results_table([row], title="Uneg: Single Test")
    return row


def run_many_tests(
    k: int = 6,
    trials: int = 20,
    seed: int = 0,
    stop_on_fail: bool = False,
) -> List[Dict[str, Any]]:
    """Run multiple randomized Uneg tests and print results with summary.

    This routine executes several random input trials, prints a
    formatted results table, and outputs a summary of pass/fail
    statistics and minimum maximum-probability observed.

    Args:
        k: Number of qubits.
        trials: Number of random trials to execute.
        seed: Random seed for reproducibility.
        stop_on_fail: Whether to stop execution after first failure.

    Returns:
        A list of dictionaries containing test results.

    Raises:
        None.
    """
    random.seed(seed)
    gate = uneg(k).to_gate()

    rows: List[Dict[str, Any]] = []
    for _ in range(trials):
        a = random.randint(1, 2**k - 1)
        row = run_uneg_once(a, k, gate=gate)
        rows.append(row)
        if stop_on_fail and not row["pass"]:
            break

    print_uneg_results_table(
        rows,
        title=f"Uneg: Many Tests (k={k}, trials={len(rows)}, seed={seed})",
    )

    passed = sum(1 for r in rows if r["pass"])
    failed = len(rows) - passed
    min_p = min((r["p_max"] for r in rows), default=None)
    print(f"\nSummary: passed={passed}, failed={failed}, min(p_max)={_fmt_float(min_p, 10)}")

    return rows


# =============================================================================
# Example usage
# =============================================================================
if __name__ == "__main__":
    k = 6
    a = 5

    run_single_test(a, k)
    run_many_tests(k=6, trials=50, seed=123, stop_on_fail=False)

Repo root: /Users/junaida/Documents/Non-Uniform-QFT
Added to sys.path: /Users/junaida/Documents/Non-Uniform-QFT/src
Uneg: Single Test
┌───┬───┬──────┬────────┬───────────────┬──────────┬───────┬───────────────┬───────┬──────────┬───────────────┬────────┐
│ # │ k │ a_in │ a_bits │     x_in      │ out_bits │ v_out │     x_out     │ exp_v │ exp_bits │     p_max     │ status │
├───┼───┼──────┼────────┼───────────────┼──────────┼───────┼───────────────┼───────┼──────────┼───────────────┼────────┤
│ 1 │ 6 │    5 │ 000101 │  0.0781250000 │   111011 │    59 │  0.9218750000 │    59 │   111011 │  1.0000000000 │   PASS │
└───┴───┴──────┴────────┴───────────────┴──────────┴───────┴───────────────┴───────┴──────────┴───────────────┴────────┘
Uneg: Many Tests (k=6, trials=50, seed=123)
┌────┬───┬──────┬────────┬───────────────┬──────────┬───────┬───────────────┬───────┬──────────┬───────────────┬────────┐
│ #  │ k │ a_in │ a_bits │     x_in      │ out_bits │ v_out │     x_out     │ exp_v │ exp_bits 